# CPS 2023 — Unemployment prediction (cross-section)

This notebook reproduces the full pipeline used in the report:
- data loading and cleaning (no weights)
- descriptive statistics tables/figures
- econometric models (LPM, logit, probit, interaction)
- prediction models (logit variants, random forest)
- cross-validation / out-of-sample evaluation
- export of LaTeX tables, figures, and a compilable `main.tex`

**Expected input file:** `cps_2023.csv` in the same folder as this notebook.

In [ ]:

# Imports
import os
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix

from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve
)
from sklearn.calibration import calibration_curve


In [ ]:

# Paths (simple: the CSV is expected next to the notebook)
DATA_PATH = "cps_2023.csv"

OUT_DIR = "outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)


In [ ]:

# Load data
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df.head())
print("\nColumns:", df.columns.tolist())


## Cleaning and construction of the modelling sample

In [ ]:

# Recode education missingness as 'None' (so we don't lose observations)
df["EDUC_LEVEL"] = df["EDUC_LEVEL"].fillna("None")

# Recode minority indicator to readable labels
df["MINORITY"] = df["MINORITY"].map({0: "Majority", 1: "Minority"})

# Create labour-force sample and UNEMP
df_lf = df[df["EMPSTATUS"].isin(["Employed", "Unemployed"])].copy()
df_lf["UNEMP"] = (df_lf["EMPSTATUS"] == "Unemployed").astype(int)

print("Labour-force sample size:", df_lf.shape)
print("Unemployment rate in labour force:", df_lf["UNEMP"].mean())


In [ ]:

# Set reference categories (first level = reference)
cat_order = {
    'SEX_F': ['Male','Female'],
    'EDUC_LEVEL': ['None','Primary','Secondary','Higher'],
    'MARITAL': ['Single','Married'],
    'RESIDENCE': ['Urban','Rural'],
    'AGE_GROUP': ['41-64','16-24','25-35','36-40'],  # reference = 41-64
    'MINORITY': ['Majority','Minority'],
    'DISABLED': ['No difficulty','Disabled']
}

for col, levels in cat_order.items():
    df_lf[col] = pd.Categorical(df_lf[col], categories=levels, ordered=False)

df_lf = df_lf.dropna(subset=list(cat_order.keys()) + ["UNEMP"])
print("Final modelling sample size:", df_lf.shape)


## Descriptive statistics tables and a cross-sectional figure

In [ ]:

# Sample composition (unweighted)
def share_table(df, col, order=None):
    vc = df[col].value_counts(dropna=False)
    if order:
        vc = vc.reindex(order)
    return (vc / vc.sum() * 100).round(2)

df_full = df.copy()
df_full["EDUC_LEVEL"] = df_full["EDUC_LEVEL"].fillna("None")
df_full["MINORITY"] = df_full["MINORITY"].map({0:"Majority (0)", 1:"Minority (1)"})

orders = {
    "Sex": ("SEX_F", ["Female","Male"]),
    "Education level": ("EDUC_LEVEL", ["Higher","None","Primary","Secondary"]),
    "Marital status": ("MARITAL", ["Married","Single"]),
    "Place of residence": ("RESIDENCE", ["Rural","Urban"]),
    "Age group": ("AGE_GROUP", ["16-24","25-35","36-40","41-64"]),
    "Minority status": ("MINORITY", ["Majority (0)","Minority (1)"]),
    "Disability status": ("DISABLED", ["Disabled","No difficulty"]),
}

rows=[]
ref_cats={
    ('Sex','Male'),
    ('Education level','None'),
    ('Marital status','Single'),
    ('Place of residence','Urban'),
    ('Age group','41-64'),
    ('Minority status','Majority (0)'),
    ('Disability status','No difficulty'),
}
for var,(col,order) in orders.items():
    shares = share_table(df_full, col, order)
    for cat, val in shares.items():
        cat_disp = f"{cat}*" if (var,cat) in ref_cats else str(cat)
        rows.append([var, cat_disp, f"{val:.2f}"])
sample_comp = pd.DataFrame(rows, columns=["Variable","Category","Share (%)"])
sample_comp


In [ ]:

# Distribution by labour-market status (% of each column, unweighted)
statuses=["Employed","Inactive","Unemployed"]
df_status = df_full.copy()
dist_rows=[]
var_orders = {
    'AGE_GROUP':['16-24','25-35','36-40','41-64'],
    'DISABLED':['Disabled','No difficulty'],
    'EDUC_LEVEL':['Higher','None','Primary','Secondary'],
    'MARITAL':['Married','Single'],
    'MINORITY':['Majority (0)','Minority (1)'],
    'RESIDENCE':['Rural','Urban'],
    'SEX_F':['Female','Male']
}

for var, cats in var_orders.items():
    for cat in cats:
        row={'Variable':var,'Category':cat}
        for st in statuses:
            subset=df_status[df_status['EMPSTATUS']==st]
            row[st] = (subset[var]==cat).mean()*100 if len(subset)>0 else np.nan
        dist_rows.append(row)

dist_df = pd.DataFrame(dist_rows)
dist_df.head()


In [ ]:

# Figure: unemployment rates by selected groups (labour force)
def unemp_rate_by(df, var, order=None):
    rates = df.groupby(var)["UNEMP"].mean()*100
    if order is not None:
        rates = rates.reindex(order)
    return rates

rates_age = unemp_rate_by(df_lf, "AGE_GROUP", ['16-24','25-35','36-40','41-64'])
rates_edu = unemp_rate_by(df_lf, "EDUC_LEVEL", ['None','Primary','Secondary','Higher'])
rates_dis = unemp_rate_by(df_lf, "DISABLED", ['No difficulty','Disabled'])
rates_min = unemp_rate_by(df_lf, "MINORITY", ['Majority','Minority'])

fig, axes = plt.subplots(2,2, figsize=(10,7))
axes = axes.flatten()

axes[0].bar(rates_age.index, rates_age.values)
axes[0].set_title("Unemployment rate by age group")

axes[1].bar(rates_edu.index, rates_edu.values)
axes[1].set_title("Unemployment rate by education")
axes[1].tick_params(axis='x', rotation=20)

axes[2].bar(rates_dis.index, rates_dis.values)
axes[2].set_title("Unemployment rate by disability status")

axes[3].bar(rates_min.index, rates_min.values)
axes[3].set_title("Unemployment rate by minority status")

for ax in axes:
    ax.set_ylabel("Percent")
    ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.tight_layout()
fig_path = os.path.join(FIG_DIR, "unemp_rates_by_group.png")
fig.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.close(fig)

fig_path


## Econometric models (LPM, logit, probit, interaction)

In [ ]:

# Baseline formula (categorical dummies; reference categories given by the first level of each Categorical)
formula = "UNEMP ~ C(SEX_F) + C(EDUC_LEVEL) + C(MARITAL) + C(RESIDENCE) + C(AGE_GROUP) + C(MINORITY) + C(DISABLED)"

# LPM (OLS)
lpm = smf.ols(formula, data=df_lf).fit()
lpm_rob = lpm.get_robustcov_results(cov_type="HC1")

# Logit
logit = smf.logit(formula, data=df_lf).fit(disp=0)

# Probit (robust)
probit = smf.probit(formula, data=df_lf).fit(disp=0, cov_type="HC1")

# Interaction logit: EDUC_LEVEL x MINORITY
formula_int = "UNEMP ~ C(SEX_F) + C(EDUC_LEVEL)*C(MINORITY) + C(MARITAL) + C(RESIDENCE) + C(AGE_GROUP) + C(DISABLED)"
logit_int = smf.logit(formula_int, data=df_lf).fit(disp=0, cov_type="HC1")

print(logit.summary().tables[1])


In [ ]:

# LPM heteroskedasticity: Breusch-Pagan test
from statsmodels.stats.diagnostic import het_breuschpagan
bp_lm, bp_pvalue, fvalue, f_pvalue = het_breuschpagan(lpm.resid, lpm.model.exog)
print("Breusch-Pagan LM:", bp_lm)
print("p-value:", bp_pvalue)


In [ ]:

# LPM residual vs fitted plot
fitted = lpm.fittedvalues
resid = lpm.resid

np.random.seed(42)
idx = np.random.choice(len(fitted), size=5000, replace=False)

fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(fitted.iloc[idx], resid.iloc[idx], s=8, alpha=0.4)
ax.axhline(0, color="grey", linewidth=1)
ax.set_xlabel("Fitted values")
ax.set_ylabel("Residuals")
ax.set_title("LPM residuals vs fitted (random sample)")
ax.grid(alpha=0.3)

path = os.path.join(FIG_DIR, "lpm_residuals_vs_fitted.png")
fig.savefig(path, dpi=200, bbox_inches="tight")
plt.close(fig)
path


## Prediction models and out-of-sample evaluation

In [ ]:

# Prepare ML design matrix (categorical only)
cat_cols = list(cat_order.keys())
X = df_lf[cat_cols].copy()
y = df_lf["UNEMP"].values

# Train/test split (pseudo-future sample)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# One-hot encoder
preprocess = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)],
    remainder="drop"
)

# Interaction feature for ML (MINORITY x EDUC_LEVEL as a combined categorical feature)
X_int = X.copy()
X_int["MIN_EDU"] = X_int["MINORITY"].astype(str) + "_" + X_int["EDUC_LEVEL"].astype(str)
X_int_cols = cat_cols + ["MIN_EDU"]

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_int[X_int_cols], y, test_size=0.2, random_state=42, stratify=y
)

preprocess_int = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), X_int_cols)],
    remainder="drop"
)

# Models
lpm_ml = Pipeline([("preprocess", preprocess), ("model", LinearRegression())])

logit_ml = Pipeline([("preprocess", preprocess),
                     ("model", LogisticRegression(max_iter=2000, solver="liblinear", C=1.0))])

logit_l1 = Pipeline([("preprocess", preprocess),
                     ("model", LogisticRegression(max_iter=5000, solver="saga", penalty="l1", C=0.2))])

logit_int_ml = Pipeline([("preprocess", preprocess_int),
                         ("model", LogisticRegression(max_iter=2000, solver="liblinear", C=1.0))])

# Random forest (light version for speed)
# We fit it on encoded matrices to access feature importances later
X_train_enc = preprocess.fit_transform(X_train)
X_test_enc = preprocess.transform(X_test)

rf = RandomForestClassifier(
    random_state=42,
    n_estimators=80,
    max_depth=8,
    min_samples_leaf=50,
    n_jobs=-1
)
rf.fit(X_train_enc, y_train)

print("Train size:", len(y_train), "Test size:", len(y_test), "Unemp rate test:", y_test.mean())


In [ ]:

# Cross-validation check (3-fold CV) for parametric models on the training set
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def eval_scores(y_true, p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return {
        "AUC": roc_auc_score(y_true, p),
        "LogLoss": log_loss(y_true, p),
        "Brier": brier_score_loss(y_true, p),
    }

models_fast = {
    "LPM (OLS)": (lpm_ml, X_train, y_train, "predict"),
    "Logit (baseline)": (logit_ml, X_train, y_train, "predict_proba"),
    "Logit (L1/LASSO)": (logit_l1, X_train, y_train, "predict_proba"),
    "Logit (interaction)": (logit_int_ml, X_train_i, y_train_i, "predict_proba"),
}

cv_rows=[]
for name, (model, Xtr, ytr, method) in models_fast.items():
    if method == "predict":
        oof = cross_val_predict(model, Xtr, ytr, cv=cv3, method="predict", n_jobs=-1)
    else:
        oof = cross_val_predict(model, Xtr, ytr, cv=cv3, method="predict_proba", n_jobs=-1)[:,1]
    scores = eval_scores(ytr, oof)
    cv_rows.append([name, scores["AUC"], scores["LogLoss"], scores["Brier"]])

cv_df = pd.DataFrame(cv_rows, columns=["Model","AUC (CV)","Log-loss (CV)","Brier (CV)"])
cv_df


In [ ]:

# Test-set evaluation for all models (including random forest)
test_rows=[]

# Fit and predict parametric models
for name, (model, Xtr, ytr, method) in models_fast.items():
    model.fit(Xtr, ytr)
    if name == "Logit (interaction)":
        Xt = X_test_i
        yt = y_test_i
    else:
        Xt = X_test
        yt = y_test

    if method == "predict":
        p = model.predict(Xt)
    else:
        p = model.predict_proba(Xt)[:,1]

    scores = eval_scores(yt, p)
    test_rows.append([name, scores["AUC"], scores["LogLoss"], scores["Brier"]])

# Random forest
p_rf = rf.predict_proba(X_test_enc)[:,1]
scores_rf = eval_scores(y_test, p_rf)
test_rows.append(["Random forest", scores_rf["AUC"], scores_rf["LogLoss"], scores_rf["Brier"]])

test_df = pd.DataFrame(test_rows, columns=["Model","AUC (Test)","Log-loss (Test)","Brier (Test)"])
test_df


In [ ]:

# ROC curves (test set)
fig, ax = plt.subplots(figsize=(7,6))

# store probabilities
preds = {
    "LPM (OLS)": np.clip(lpm_ml.predict(X_test), 1e-6, 1-1e-6),
    "Logit (baseline)": logit_ml.fit(X_train,y_train).predict_proba(X_test)[:,1],
    "Logit (interaction)": logit_int_ml.fit(X_train_i,y_train_i).predict_proba(X_test_i)[:,1],
    "Random forest": p_rf
}

for name, p in preds.items():
    yt = y_test_i if name=="Logit (interaction)" else y_test
    fpr, tpr, _ = roc_curve(yt, p)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(yt,p):.3f})")

ax.plot([0,1],[0,1], linestyle="--", color="grey", linewidth=1)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves on the test set")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

path = os.path.join(FIG_DIR, "roc_models.png")
fig.savefig(path, dpi=200, bbox_inches="tight")
plt.close(fig)
path


In [ ]:

# Calibration curves (test set)
fig, ax = plt.subplots(figsize=(7,6))
for name in ["Logit (baseline)","Logit (interaction)","Random forest"]:
    p = preds[name]
    yt = y_test_i if name=="Logit (interaction)" else y_test
    frac_pos, mean_pred = calibration_curve(yt, p, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", label=name)

ax.plot([0,1],[0,1], linestyle="--", color="grey", linewidth=1)
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed unemployment rate")
ax.set_title("Calibration curves (test set)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

path = os.path.join(FIG_DIR, "calibration_models.png")
fig.savefig(path, dpi=200, bbox_inches="tight")
plt.close(fig)
path


In [ ]:

# Distribution of predicted probabilities (random forest)
fig, ax = plt.subplots(figsize=(7,5))
ax.hist(p_rf[y_test==0], bins=30, alpha=0.6, label="Employed (0)")
ax.hist(p_rf[y_test==1], bins=30, alpha=0.6, label="Unemployed (1)")
ax.set_xlabel("Predicted probability of unemployment")
ax.set_ylabel("Count")
ax.set_title("Distribution of predicted probabilities (Random forest, test set)")
ax.legend()
ax.grid(axis="y", alpha=0.3)

path = os.path.join(FIG_DIR, "pred_proba_distribution.png")
fig.savefig(path, dpi=200, bbox_inches="tight")
plt.close(fig)
path


## Example predictions + confidence intervals (logit)

In [ ]:

# Delta-method CI for logit predicted probabilities
design_info = logit.model.data.design_info

def predict_logit_ci(res, new_df, alpha=0.05):
    X_new = dmatrix(design_info, new_df, return_type="dataframe")
    beta = res.params.values
    cov = res.cov_params().values
    lin = np.dot(X_new, beta)
    p = 1/(1+np.exp(-lin))
    grad = (p*(1-p))[:,None] * X_new.values
    var_p = np.einsum("ij,jk,ik->i", grad, cov, grad)
    se = np.sqrt(var_p)
    z = stats.norm.ppf(1-alpha/2)
    lo = np.clip(p - z*se, 0, 1)
    hi = np.clip(p + z*se, 0, 1)
    return pd.DataFrame({"p_hat":p, "lower":lo, "upper":hi})

profiles = pd.DataFrame([
    {"SEX_F":"Male","EDUC_LEVEL":"None","MARITAL":"Single","RESIDENCE":"Urban","AGE_GROUP":"16-24","MINORITY":"Minority","DISABLED":"Disabled"},
    {"SEX_F":"Female","EDUC_LEVEL":"Higher","MARITAL":"Married","RESIDENCE":"Urban","AGE_GROUP":"41-64","MINORITY":"Majority","DISABLED":"No difficulty"},
    {"SEX_F":"Male","EDUC_LEVEL":"Secondary","MARITAL":"Single","RESIDENCE":"Rural","AGE_GROUP":"25-35","MINORITY":"Minority","DISABLED":"No difficulty"},
    {"SEX_F":"Female","EDUC_LEVEL":"Primary","MARITAL":"Single","RESIDENCE":"Urban","AGE_GROUP":"16-24","MINORITY":"Majority","DISABLED":"No difficulty"},
])

for col, levels in cat_order.items():
    profiles[col] = pd.Categorical(profiles[col], categories=levels)

ci = predict_logit_ci(logit, profiles)

# Random forest point predictions for the same profiles
profiles_enc = preprocess.transform(profiles[cat_cols])
rf_p = rf.predict_proba(profiles_enc)[:,1]

out = profiles.copy()
out["logit_p"] = ci["p_hat"]
out["logit_CI"] = ["[{:.3f}, {:.3f}]".format(l,u) for l,u in zip(ci["lower"], ci["upper"])]
out["rf_p"] = rf_p
out


## Optional: evaluate on CPS 2025 if you have it

If you later obtain a file `cps_2025.csv` with the same columns as `cps_2023.csv`, you can run the block below to produce a true 2025 evaluation. Otherwise it will simply skip.

In [ ]:

CPS_2025_PATH = "cps_2025.csv"
if os.path.exists(CPS_2025_PATH):
    df25 = pd.read_csv(CPS_2025_PATH)
    df25["EDUC_LEVEL"] = df25["EDUC_LEVEL"].fillna("None")
    df25["MINORITY"] = df25["MINORITY"].map({0:"Majority",1:"Minority"})
    df25 = df25[df25["EMPSTATUS"].isin(["Employed","Unemployed"])].copy()
    df25["UNEMP"] = (df25["EMPSTATUS"]=="Unemployed").astype(int)
    for col, levels in cat_order.items():
        df25[col] = pd.Categorical(df25[col], categories=levels)
    df25 = df25.dropna(subset=list(cat_order.keys())+["UNEMP"])
    X25 = df25[cat_cols]
    y25 = df25["UNEMP"].values
    X25_enc = preprocess.transform(X25)

    p25 = rf.predict_proba(X25_enc)[:,1]
    print("2025 AUC:", roc_auc_score(y25, p25))
    print("2025 LogLoss:", log_loss(y25, np.clip(p25,1e-6,1-1e-6)))
else:
    print("No cps_2025.csv found — skipping.")
